In [5]:
# !pip install opencv-python scikit-learn

# This is all from here:
# https://lab.slv.vic.gov.au/resources/introduction-to-k-means-clustering

import cv2 as cv
import requests
import numpy as np
from PIL import Image
from sklearn.cluster import KMeans
from IPython.display import Image, display
import pandas as pd 
from sudulunu.helpers import pp, dumper

In [6]:
def rgb_to_hex(rgb_array):
    hex_colors = []
    for rgb in rgb_array:
        r, g, b = [int(round(val)) for val in rgb]
        hex_color = f'#{r:02x}{g:02x}{b:02x}'
        hex_colors.append(hex_color)
    return hex_colors

def sorter(hex_colors):
    def hex_to_brightness(hex_code):
        hex_code = hex_code.lstrip('#')
        r, g, b = tuple(int(hex_code[i:i+2], 16) for i in (0, 2, 4))
        brightness = 0.299 * r + 0.587 * g + 0.114 * b
        return brightness

    # Sort colors by brightness (descending for light to dark)
    return sorted(hex_colors, key=hex_to_brightness, reverse=True)

In [7]:
# ### This is the first test

# pathos = '/Users/josh/Github/site/static/blueyellow.jpg'

# with open(pathos, "rb") as f:
#     file_bytes = f.read()

# img_array = np.frombuffer(file_bytes, np.uint8)
# # img = Image.open(pathos)
# # img_array = np.array(img)

# img = cv.imdecode(img_array, -1)
# img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
# # img = cv.imdecode(img_array, cv.IMREAD_COLOR)

# width, height = 300, 300
# img = cv.resize(img, (width, height), interpolation=cv.INTER_AREA)
# # display(img)
# # from PIL import Image
# # pil_img = Image.fromarray(img)
# # display(pil_img)

# no_of_clusters = 5
# pixels = img.reshape(-1, 3)

# cluster_model = KMeans(n_clusters=no_of_clusters)
# clusters = cluster_model.fit(pixels)

# def rgb_to_hex(rgb_array):
#     """Convert an array of RGB values to a list of hex color codes."""
#     hex_colors = []
#     for rgb in rgb_array:
#         # Make sure values are integers between 0-255
#         r, g, b = [int(round(val)) for val in rgb]
#         # Convert to hex format (#RRGGBB)
#         hex_color = f'#{r:02x}{g:02x}{b:02x}'
#         hex_colors.append(hex_color)
#     return hex_colors

# palette_colors = clusters.cluster_centers_
# hex_colors = rgb_to_hex(palette_colors)

# print(hex_colors)

# # clusters.cluster_centers_

# # palette = np.zeros((50, width, 3), np.uint8)
# # steps = width / clusters.cluster_centers_.shape[0]
# # for idx, centers in enumerate(clusters.cluster_centers_):
# #   palette[:, int(idx * steps) : (int((idx + 1) * steps)), :] = centers

# # # palette

# # from PIL import Image
# # pil_img = Image.fromarray(palette)
# # display(pil_img)

In [8]:
### This is the actual

frame = pd.read_csv('/Users/josh/Github/site/python/scrap/together.csv')
# ['Date', 'Title', 'img_path', 'Caption', 'Colours', 'Style', 
#  'Subject', 'Keywords', 'Category', 'img_alt', 'Width', 'Height']

base = '/Users/josh/Github/site/static/images/'
out_path = '/Users/josh/Github/site/python/scrap'

# frame = frame[:1]

listo = []

for index, row in frame.iterrows():
    pathos = base + row['img_path']

    with open(pathos, "rb") as f:
        file_bytes = f.read()

    img_array = np.frombuffer(file_bytes, np.uint8)

    img = cv.imdecode(img_array, -1)
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

    width, height = 300, 300
    img = cv.resize(img, (width, height), interpolation=cv.INTER_AREA)

    no_of_clusters = 5
    pixels = img.reshape(-1, 3)

    cluster_model = KMeans(n_clusters=no_of_clusters)
    clusters = cluster_model.fit(pixels)

    palette_colors = clusters.cluster_centers_
    hex_colors = sorter(rgb_to_hex(palette_colors))

    record = {"Date": row['Date'], 'img_path': row['img_path']}
    

    for i in range(0, len(hex_colors)):
        # print(i)
        # print(hex_colors[i])
        record[i] = hex_colors[i]
    
    listo.append(record)

outty = pd.DataFrame.from_records(listo)
outty.sort_values(by='Date', ascending=True, inplace=True)

pp(outty)

dumper(out_path, 'colours', outty)
    # print(hex_colors)

# outty.to_json('/Users/josh/Github/site/static/colours.json', orient='records')
outty.to_json('/Users/josh/Github/site/src/lib/data/colours.json', orient='records')

           Date                                           img_path        0  \
365  2019-07-24         134ScribblingfromDublincastle--------3.jpg  #d3d2da   
364  2019-07-27              132Sawaguywithpigtailsinhisbeardf.jpg  #e0ad7f   
363  2019-07-29                131Warsawisdamnprettytbh------0.jpg  #e6cab0   
362  2019-07-30                     129Beforeandafter--------b.jpg  #e3c6ae   
361  2019-08-03  127KrakowKathedral--therearetoomanybuildingsto...  #c7cbce   
..          ...                                                ...      ...   
4    2025-05-07  bafkreidt4dywgkpgutce3ej2clieqwa6bn3iq7an7ez6q...  #939daa   
3    2025-05-10  bafkreih7gjb5gdshhw6a5sdd2vkitdlv3ox2jw5ega4co...  #83afde   
2    2025-05-12  bafkreiferxq3oxvfuur3upd4qqjd4l25esllnwhmt5oie...  #ecc8a9   
1    2025-05-13  bafkreiauxw5n6prhjmphzu3jioktkxmqt5p65jn5cuhyi...  #f0b55e   
0    2025-05-18  bafkreiclttl5ligykag5vnflkrs62fj47kdxcidc53skl...  #c1b9ad   

           1        2        3        4  
365  #caa